In [36]:
!pip install pyspark


In [37]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, to_date

# 1. Initialize Spark Session
spark = SparkSession.builder \
    .appName("Day1_Clickstream_Optimization") \
    .getOrCreate()

# 2. Create Mock Data (Simulating millions of incoming rows)
data = [
    ("U1001", "2026-07-12 10:00:00", "click", 0.0),
    ("U1002", "2026-07-12 10:05:00", "purchase", 150.50),
    (None, "2026-07-12 10:06:00", "view", 0.0), # Null row to drop
    ("U1001", "2026-07-12 11:00:00", "purchase", 45.25),
    ("U1003", "2026-07-12 12:15:00", "click", 0.0)
]
columns = ["user_id", "event_time", "action", "amount"]
raw_df = spark.createDataFrame(data, schema=columns)

# 3. Apply Transformations (Filter and Add Date)
cleaned_df = raw_df.filter(col("user_id").isNotNull()) \
                   .withColumn("event_date", to_date(col("event_time")))

# 4. Write data to "Cloud Storage" (Simulated locally in Colab as Parquet + Partitioned)
# In production, 'optimized_clickstream' would be an S3 or GCS path like 's3://my-bucket/parquet_data/'
cleaned_df.write \
    .mode("overwrite") \
    .partitionBy("event_date") \
    .parquet("optimized_clickstream")

print("Day 1 Challenge complete: Data successfully optimized and written!")


Day 1 Challenge complete: Data successfully optimized and written!


In [38]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, when

# 1. Start our Spark Engine
spark = SparkSession.builder.appName("Day2_Data_Warehouse_Prep").getOrCreate()

# 2. Input some dirty data (simulating system errors sending negative amounts)
day2_data = [
    ("U1001", "click", 0.0),
    ("U1002", "purchase", 250.0),
    ("U1003", "purchase", -99.0),  # Malformed row!
    ("U1001", "purchase", 50.0),
    ("U1002", "click", -5.0)       # Malformed row!
]
columns = ["user_id", "action", "amount"]
df = spark.createDataFrame(day2_data, schema=columns)

# 3. FIX THE MALFORMED DATA
# If amount is less than 0, change it to 0.0. Otherwise, keep the original amount.
cleaned_df = df.withColumn("amount", when(col("amount") < 0, 0.0).otherwise(col("amount")))

# 4. AGGREGATE THE DATA FOR THE WAREHOUSE
# Group by user_id and sum up all their clean amounts
final_df = cleaned_df.groupBy("user_id").sum("amount")

# 5. Show the final clean business report
final_df.show()

+-------+-----------+
|user_id|sum(amount)|
+-------+-----------+
|  U1001|       50.0|
|  U1002|      250.0|
|  U1003|        0.0|
+-------+-----------+



In [39]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, broadcast

# 1. Initialize Spark Engine
spark = SparkSession.builder.appName("Day3_Broadcast_Join").getOrCreate()

# 2. Large DataFrame (Simulating millions of rows of sales)
sales_data = [
    ("U1001", 150.00),
    ("U1002", 45.25),
    ("U1001", 30.00),
    ("U1003", 99.99)
]
sales_df = spark.createDataFrame(sales_data, ["user_id", "price"])

# 3. Small Lookup DataFrame (Only a few rows mapping IDs to Names)
user_profiles_data = [
    ("U1001", "Alice"),
    ("U1002", "Bob"),
    ("U1003", "Charlie")
]
users_df = spark.createDataFrame(user_profiles_data, ["user_id", "user_name"])

# --- DAY 3 ARCHITECTURAL JOIN ---
# We use broadcast() on the small table to prevent a network shuffle!
optimized_joined_df = sales_df.join(broadcast(users_df), "user_id")

# Show the results
optimized_joined_df.show()


+-------+-----+---------+
|user_id|price|user_name|
+-------+-----+---------+
|  U1001|150.0|    Alice|
|  U1002|45.25|      Bob|
|  U1001| 30.0|    Alice|
|  U1003|99.99|  Charlie|
+-------+-----+---------+



In [40]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col

# 1. Initialize Spark Engine
spark = SparkSession.builder.appName("Day5_Memory_Management").getOrCreate()

# 2. Giant Mock Dataset (Imagine millions of rows with many columns)
# We have user data, but we only care about user_id and active status today.
giant_data = [
    ("U1001", "Alice", "USA", "Premium", "Active"),
    ("U1002", "Bob", "India", "Free", "Inactive"),
    ("U1003", "Charlie", "UK", "Premium", "Active"),
    ("U1004", "David", "Canada", "Free", "Active")
]
columns = ["user_id", "name", "country", "subscription_type", "status"]
large_df = spark.createDataFrame(giant_data, schema=columns)

# --- DAY 5 ARCHITECTURAL MEMORY OPTIMIZATION ---
# Instead of keeping all 5 columns in memory, we immediately SELECT
# only the 2 columns we need, and FILTER out inactive users.

optimized_df = large_df.select("user_id", "status") \
                        .filter(col("status") == "Active")

# Show the results
optimized_df.show()


+-------+------+
|user_id|status|
+-------+------+
|  U1001|Active|
|  U1003|Active|
|  U1004|Active|
+-------+------+



In [41]:
from pyspark.sql import SparkSession

# 1. Initialize our Spark Engine
spark = SparkSession.builder.appName("Day6_Parquet_Compression").getOrCreate()

# 2. Raw Text Data (Imagine millions of rows of text taking up huge space)
raw_text_data = [
    ("U1001", "Click daily login banner on home screen layout"),
    ("U1002", "Purchase checkout completed via mobile application portal"),
    ("U1003", "View items list under clearance sale banner section")
]
columns = ["user_id", "user_activity_log"]
df = spark.createDataFrame(raw_text_data, schema=columns)

# --- DAY 6 ARCHITECTURAL COMPRESSION ---
# 3. We save this text data as a highly compressed Parquet file.
# Spark automatically strips out the empty space and encodes it into binary.
df.write \
  .mode("overwrite") \
  .parquet("compressed_activity_logs")

print("Day 6 complete: Text data vacuum-sealed into Parquet!")

Day 6 complete: Text data vacuum-sealed into Parquet!


In [42]:
from pyspark.sql import SparkSession
from pyspark.sql.types import StructType, StructField, StringType, IntegerType
from pyspark.sql.functions import col

# 1. Initialize our Spark Engine
spark = SparkSession.builder.appName("Day7_Schema_Enforcement_Fixed").getOrCreate()

# 2. Define the Strict Blueprint (Schema)
clean_blueprint = StructType([
    StructField("user_id", StringType(), True),
    StructField("age", IntegerType(), True) # Age MUST be an integer!
])

# 3. Create dummy data as RDD text strings (mimicking reading from a raw cloud log file)
raw_lines = spark.sparkContext.parallelize([
    "U1001,25",
    "U1002,thirty" # Broken string value instead of integer
])

# 4. Parse text lines into columns split by comma
parsed_rdd = raw_lines.map(lambda line: line.split(","))

# 5. Enforce schema by mapping string values to schema types safely
# Spark will now handle the type conversion dynamically, turning bad data to null
def safe_cast(row):
    user_id = row[0]
    try:
        age = int(row[1])
    except ValueError:
        age = None # Turn the broken text into null safely!
    return (user_id, age)

final_rdd = parsed_rdd.map(safe_cast)

# 6. Load into DataFrame with our schema
df = spark.createDataFrame(final_rdd, schema=clean_blueprint)

# Show the results to see how Spark handles the bad row
df.show()

+-------+----+
|user_id| age|
+-------+----+
|  U1001|  25|
|  U1002|NULL|
+-------+----+



In [43]:
from pyspark.sql import SparkSession

# 1. Initialize our Spark Engine
# In production, we configure this to include the warehouse connector extensions
spark = SparkSession.builder \
    .appName("Day8_Data_Warehouse_Load") \
    .getOrCreate()

# 2. Fully Cleaned, Modeled Data ready for Business Analysts
# (Imagine millions of rows ready for dbt to transform later)
warehouse_ready_data = [
    ("U1001", "Premium", 150.00),
    ("U1003", "Premium", 99.99),
    ("U1004", "Free", 0.00)
]
columns = ["user_id", "tier", "total_spent"]
final_analytics_df = spark.createDataFrame(warehouse_ready_data, schema=columns)

# --- DAY 8 ARCHITECTURAL WAREHOUSE WRITE ---
# This is the exact pattern used to write to databases.
# In a real setup, format("parquet") becomes format("bigquery") or format("redshift")
final_analytics_df.write \
    .format("parquet") \
    .mode("overwrite") \
    .option("table", "company_analytics.user_revenue_summary") \
    .save("cloud_warehouse_shelf")

print("Day 8 complete: Data safely loaded into the Cloud Warehouse!")


Day 8 complete: Data safely loaded into the Cloud Warehouse!


In [45]:
import time

# --- DAY 9 ARCHITECTURAL AUTOMATION (AIRFLOW SIMULATION) ---

def task_1_check_cloud_storage():
    print("[Airflow Task 1]: Checking AWS S3 / Google Cloud Storage for new files...")
    time.sleep(1) # Simulating a quick network check
    print("SUCCESS: New raw data logs found!")
    return True

def task_2_run_pyspark_pipeline():
    print("[Airflow Task 2]: Triggering Apache Spark cluster engine...")
    time.sleep(2) # Simulating Spark cleaning out mud/negative values
    print("SUCCESS: Spark successfully vacuum-sealed data into Parquet!")
    return True

def task_3_load_to_data_warehouse():
    print("[Airflow Task 3]: Pushing clean files to Cloud Data Warehouse...")
    time.sleep(1) # Simulating the data warehouse connector load
    print("SUCCESS: Data is live on supermarket shelves! Analysts can run SQL.")
    return True

# --- THE MASTER SUPERVISOR EXECUTION ---
print("--- ⏰ 2:00 AM: AIRFLOW AUTOMATION TRIGGERED ⏰ ---")

# Fixed the syntax here by adding the missing 'if' on the third task
if task_1_check_cloud_storage():
    if task_2_run_pyspark_pipeline():
        if task_3_load_to_data_warehouse():
            print("\n--- ✅ PIPELINE RUN COMPLETE: Everything succeeded automatically! ---")

--- ⏰ 2:00 AM: AIRFLOW AUTOMATION TRIGGERED ⏰ ---
[Airflow Task 1]: Checking AWS S3 / Google Cloud Storage for new files...
SUCCESS: New raw data logs found!
[Airflow Task 2]: Triggering Apache Spark cluster engine...
SUCCESS: Spark successfully vacuum-sealed data into Parquet!
[Airflow Task 3]: Pushing clean files to Cloud Data Warehouse...
SUCCESS: Data is live on supermarket shelves! Analysts can run SQL.

--- ✅ PIPELINE RUN COMPLETE: Everything succeeded automatically! ---


In [46]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, when

# 1. Initialize the Heavy Spark Production Engine
spark = SparkSession.builder \
    .appName("Production_BigData_Pipeline_Capstone") \
    .getOrCreate()

print("--- 🚀 STAGE 1: SPARK PRODUCTION ENGINE ONLINE ---")

# 2. Raw Corrupt Input Data arriving from the Cloud Tank
raw_dirty_logs = [
    ("U1001", "2026-07-12", "purchase", 150.0), # Good data
    (None,    "2026-07-12", "click", 0.0),       # Bad data (Missing user_id)
    ("U1002", "2026-07-12", "purchase", -45.0),  # Bad data (Negative amount)
    ("U1003", "2026-07-13", "purchase", 89.99),  # Good data (Different date)
    ("U1001", "2026-07-13", "click", -5.0)       # Bad data (Negative amount)
]
columns = ["user_id", "event_date", "action", "amount"]
raw_df = spark.createDataFrame(raw_dirty_logs, schema=columns)

print("--- 📊 STAGE 2: RAW UNPROCESSED LOGS INGESTED ---")

# 3. Data Cleansing & Transformation Layer
# Step A: Drop rows where user_id is missing (null)
filtered_df = raw_df.filter(col("user_id").isNotNull())

# Step B: Fix negative numbers (change less than 0 to 0.0)
cleaned_production_df = filtered_df.withColumn(
    "amount",
    when(col("amount") < 0, 0.0).otherwise(col("amount"))
)

print("--- 🧼 STAGE 3: DATA CLEANSING & TRANSFORMATION COMPLETE ---")

# 4. Columnar Storage & Partitioning Layer
# Vacuum-seal data into Parquet format and organize into date folders
cleaned_production_df.write \
    .mode("overwrite") \
    .partitionBy("event_date") \
    .parquet("production_lakehouse_storage")

print("--- 🗄️ STAGE 4: COMPRESSED PARQUET DATA PARTITIONED TO LAKEHOUSE ---")
print("\n🎉 CAPSTONE PROJECT PIPELINE EXECUTED SUCCESSFULLY! 🎉")

# Show what the final clean warehouse data looks like
print("\nFinal clean data summary sent to Cloud Warehouse:")
cleaned_production_df.show()

--- 🚀 STAGE 1: SPARK PRODUCTION ENGINE ONLINE ---
--- 📊 STAGE 2: RAW UNPROCESSED LOGS INGESTED ---
--- 🧼 STAGE 3: DATA CLEANSING & TRANSFORMATION COMPLETE ---
--- 🗄️ STAGE 4: COMPRESSED PARQUET DATA PARTITIONED TO LAKEHOUSE ---

🎉 CAPSTONE PROJECT PIPELINE EXECUTED SUCCESSFULLY! 🎉

Final clean data summary sent to Cloud Warehouse:
+-------+----------+--------+------+
|user_id|event_date|  action|amount|
+-------+----------+--------+------+
|  U1001|2026-07-12|purchase| 150.0|
|  U1002|2026-07-12|purchase|   0.0|
|  U1003|2026-07-13|purchase| 89.99|
|  U1001|2026-07-13|   click|   0.0|
+-------+----------+--------+------+

